# Step 4b — Refined taxonomy (v0.3 proposal) and codebook: expression category separated from bias judgment

**Purpose.** Revise the schema of Step 4 (v0.2) as requested: (1) record **what the wording is** (the linguistic expression category) separately from **whether it is biased** (the judgment); (2) add **G1** (non-restrictive gender listing, e.g. "Female/Male") and **U** (unclear) and redefine **INC**; (3) give every code a rule: meaning, when to apply, when not to apply, and a real corpus example.

**Status.** This is a **proposal for review**. The pilot taxonomy is the one supplied by the project owner. Steps 5, 5a and 6 still use the v0.2 codes and are **not changed**; section 3 maps v0.2 to v0.3 so the two can be compared. No human has labelled anything, and no label below is a final judgment.

**Two fields instead of one.**

| Field | Question it answers | Filled by |
|---|---|---|
| `expression_code` / `expression_category` | What kind of wording is this span? | the lexicon/rules (candidate) |
| `primary_classification` | Is the expression type explicit or implicit (or neither)? | follows from the code |
| `bias_judgment_prior` | What is the default stance for this expression type? | defined once per code (below) |
| `final_bias_judgment` | Is this span, in this advertisement, biased? | **left `pending`**; needs a reviewer with the full ad |

**Inputs.** `outputs/s5_preannotation/preannotation_spans.csv` (v0.2 candidates), `outputs/analysis/units.csv`.
**Outputs** (`outputs/s4b_taxonomy/`): `taxonomy_v0.3.json`, `codebook_v0.3.md`, `crosswalk_v02_to_v03.csv`, `crosswalk_summary.csv`, `lexicon_gap_probe.csv`, `validation_checks.csv`.

In [1]:
import json, re
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 220)
OUT = Path("outputs") / "s4b_taxonomy"
OUT.mkdir(parents=True, exist_ok=True)

units = pd.read_csv("outputs/analysis/units.csv", encoding="utf-8-sig").set_index("unit_id", drop=False)
spans = pd.read_csv("outputs/s5_preannotation/preannotation_spans.csv", encoding="utf-8-sig")
print(len(units), "units;", len(spans), "v0.2 candidate span rows")

7971 units; 1218 v0.2 candidate span rows


## 1. Taxonomy v0.3 (pilot)

`bias_judgment_prior` uses the wording supplied by the project owner. It is a **default stance for the expression type**, not a verdict on any span.

In [2]:
TAXONOMY = [
    # code, category, definition, bias_judgment_prior, primary_classification
    ("E1", "Gender restriction or preference", "States that a gender is required or preferred.", "Explicit", "explicit"),
    ("E2", "Gender-marked occupational wording", "Gender-specific occupational term, such as 'saleslady' or 'foreman'.", "Context-dependent", "explicit"),
    ("E3", "Generic gendered pronoun", "Uses 'he' or 'she' as a generic reference to the position holder.", "Context-dependent", "explicit"),
    ("G1", "Non-restrictive gender listing", "Mentions gender categories without a stated preference, e.g. 'Female/Male'.", "Not automatically biased", "none"),
    ("I1", "Masculine-coded expression", "Agentic or masculine-associated wording supported by a defined lexicon or contextual evidence.", "Assess context", "implicit"),
    ("I2", "Feminine-coded expression", "Communal or feminine-associated wording supported by a defined lexicon or contextual evidence.", "Assess context", "implicit"),
    ("INC", "Inclusive expression", "Explicitly expresses inclusive applicant eligibility.", "Not biased on its own", "none"),
    ("P", "Polysemantic / contextual near miss", "Potential lexical match that is non-biased in context.", "Not biased", "none"),
    ("U", "Unclear expression", "Cannot be classified reliably with available context.", "Requires review", "unclear"),
]
taxonomy = pd.DataFrame(TAXONOMY, columns=["code", "category", "definition", "bias_judgment_prior", "primary_classification"])
display(taxonomy)
FINAL_JUDGMENT_VALUES = ["pending", "biased", "not_biased", "undetermined"]
json.dump({"version": "0.3-pilot", "status": "proposal for review; not piloted; no human labels",
           "fields": {"expression_code": "linguistic category", "bias_judgment_prior": "default stance per expression type", "final_bias_judgment": FINAL_JUDGMENT_VALUES},
           "labels": taxonomy.to_dict("records")}, open(OUT / "taxonomy_v0.3.json", "w", encoding="utf-8"), indent=1)

,code,category,definition,bias_judgment_prior,primary_classification
0,E1,Gender restriction or preference,States that a gender is required or preferred.,Explicit,explicit
1,E2,Gender-marked occupational wording,"Gender-specific occupational term, such as 'saleslady' or 'foreman'.",Context-dependent,explicit
2,E3,Generic gendered pronoun,Uses 'he' or 'she' as a generic reference to the position holder.,Context-dependent,explicit
3,G1,Non-restrictive gender listing,"Mentions gender categories without a stated preference, e.g. 'Female/Male'.",Not automatically biased,none
4,I1,Masculine-coded expression,Agentic or masculine-associated wording supported by a defined lexicon or contextual evidence.,Assess context,implicit
5,I2,Feminine-coded expression,Communal or feminine-associated wording supported by a defined lexicon or contextual evidence.,Assess context,implicit
6,INC,Inclusive expression,Explicitly expresses inclusive applicant eligibility.,Not biased on its own,none
7,P,Polysemantic / contextual near miss,Potential lexical match that is non-biased in context.,Not biased,none
8,U,Unclear expression,Cannot be classified reliably with available context.,Requires review,unclear


## 2. Codebook

Every code has the same six elements. Corpus examples are **looked up in the data by `unit_id`** and checked in section 5: the span must reproduce its text in the unit, and the example's v0.2 candidate must be the one the rules produced. Where the corpus has no instance, the codebook says so instead of inventing one.

Spans are character offsets in the unit's normalized text (`text_norm`, the lowercase, whitespace-normalised form from Steps 1–3, same length as the masked text).

In [3]:
# (code): dict(apply, not_apply, examples=[(unit_id, span_text_as_in_unit, why)], justification, rule_note)
CB = {
 "E1": dict(
   apply="The span states a gender as an applicant requirement or preference ('Female', 'Male only', 'preferably female', 'for male'), and it functions as a condition on who may apply.",
   not_apply="The gender mention is a listing of both genders without a preference (G1). The gender word belongs to a job title (E2) or a pronoun (E3).",
   examples=[("row-0038-s002", "Female", "Cashier; the standalone designation is presented as an applicant qualification"),
             ("row-0003-s002", None, None)],
   justification="A standalone gender designation under qualifications is an eligibility condition.",
 ),
 "E2": dict(
   apply="A job title or role noun that itself marks gender ('saleslady', 'foreman', 'watchman', 'salesman').",
   not_apply="The noun merely ends in -man/-men but is not a person term ('human', 'manager'). The word is a gender restriction on applicants (E1).",
   examples=[("row-0176-s001", "Saleslady", "job title marks the role as female"), ("row-0131-s001", "Foreman", "job title with a male-marked noun")],
   justification="The gender is carried by the occupational term, not by a stated requirement; whether it is biased depends on how the ad uses it, so the judgment prior is context-dependent.",
 ),
 "E3": dict(
   apply="'he', 'she', 'his' or 'her' refers generically to the position holder, not to a named or specific person.",
   not_apply="The pronoun refers to a specific named person. Both pronouns are given together as a neutral pair (decision pending, see section 6).",
   examples=[("row-0716-s006", "he", "'he's area of responsibility' refers to the post holder")],
   justification="A gendered pronoun used for any holder of the role implies a default gender; judgment depends on context.",
 ),
 "G1": dict(
   apply="The text names both gender categories with no preference expressed ('Female/Male', 'Male or Female', 'he/she' as a listing).",
   not_apply="Only one gender is named (E1). A statement of inclusive eligibility without naming gender categories (INC).",
   examples=[("row-0637-s003", "Female or Male", "both categories listed, no preference stated")],
   justification="A listing does not state that either gender is preferred, so it is not claimed to equal 'Female applicants only'; it is not automatically biased.",
 ),
 "I1": dict(
   apply="A word from the defined masculine-coded (agentic) lexicon is present. The code records the wording only; it does not say the ad is biased.",
   not_apply="The word is an ordinary role noun or topic ('analyst', 'analysis') - consider P. The word is only part of a technical term.",
   examples=[("row-0568-s007", "Analytical", "lexicon word in a qualification line")],
   justification="Lexicon match only; the judgment needs the full ad and is left to review.",
 ),
 "I2": dict(
   apply="A word from the defined feminine-coded (communal) lexicon is present. The code records the wording only.",
   not_apply="The word is an ordinary role noun ('support' in 'IT support') - consider P.",
   examples=[("row-0317-s009", "interpersonal", "lexicon word in a qualification line")],
   justification="Lexicon match only; the judgment needs the full ad and is left to review.",
 ),
 "INC": dict(
   apply="The ad explicitly states that applicants of any gender are eligible (for example 'regardless of gender', 'open to all genders').",
   not_apply="Gender categories are listed as in 'Female/Male' (G1). General welcoming wording that does not mention gender ('open to all college graduates').",
   examples=[],
   justification="In the v0.2 notebooks this code covered 'Female/Male' listings; those are now G1. No corpus unit with explicit all-gender eligibility wording was found by the probe in section 4, so no example is given.",
 ),
 "P": dict(
   apply="A word that matches the gender lexicon but is not about gender in this context ('MS Office', the honorific 'Ms.' for a contact person).",
   not_apply="The word refers to the applicant's gender or marks the role by gender.",
   examples=[("row-0397-s005", "MS", "'MS' here is Microsoft"), ("row-0029-s009", "Ms", "honorific for a contact person")],
   justification="Lexical match with no gender meaning in context.",
 ),
 "U": dict(
   apply="A gender word is present but the rules or the available unit text cannot place it (for example a gender word in a unit with no clear function, or a possible technical term).",
   not_apply="The span fits another code. Use U rather than guessing.",
   examples=[("row-0012-s002", "Male", "a whole-unit gender word in a block without a qualification header; probably E1, not decided"),
             ("row-0692-s004", "MAN", "'MAN' as a network type; probably P, not decided")],
   justification="Kept for review; the judgment prior is 'requires review'.",
 ),
}
cb_rows = []
for code_, d in CB.items():
    for uid, sp, why in d["examples"] or [(None, None, None)]:
        if code_ == "E1" and uid == "row-0003-s002":
            continue
        cb_rows.append((code_, uid, sp, why))
print(len(cb_rows), "example rows")

12 example rows


In [4]:
def find_example(code_, uid, text):
    # v0.2 code that the rules gave this span (for the check) and offsets
    V2 = {"E1": "E1", "E2": "E2", "E3": "E4_or_E5", "G1": "INC", "I1": "I1", "I2": "I2", "P": "P", "U": "needs_review"}
    r = spans[(spans["unit_id"] == uid) & (spans["span_text"] == text) & (spans["v02_candidate"] == V2[code_])]
    return None if r.empty else r.iloc[0]

examples = []
for code_, uid, sp, why in cb_rows:
    if uid is None:
        continue
    r = find_example(code_, uid, sp)
    examples.append({
        "code": code_, "unit_id": uid, "record_id": units.loc[uid, "record_id"], "component": units.loc[uid, "component"],
        "unit_text": units.loc[uid, "text_masked"], "span": sp, "span_start": None if r is None else int(r["span_start"]),
        "span_end": None if r is None else int(r["span_end"]), "found_as_v02_candidate": r is not None, "why": why,
    })
ex = pd.DataFrame(examples)
ex["span_in_text_norm"] = [
    (units.loc[u, "text_norm"][int(a):int(b)] if pd.notna(a) else None) for u, a, b in zip(ex["unit_id"], ex["span_start"], ex["span_end"])
]
display(ex[["code", "unit_id", "component", "unit_text", "span", "span_start", "span_end", "found_as_v02_candidate", "span_in_text_norm"]])

,code,unit_id,component,unit_text,span,span_start,span_end,found_as_v02_candidate,span_in_text_norm
0,E1,row-0038-s002,qualification,Female,Female,0,6,True,female
1,E2,row-0176-s001,job_title,Saleslady,Saleslady,0,9,True,saleslady
2,E2,row-0131-s001,job_title,Foreman,Foreman,0,7,True,foreman
3,E3,row-0716-s006,qualification,Submit Reports Daily & Weekly for sites under he’s area of responsibility,he,46,48,True,he
4,G1,row-0637-s003,qualification,Female or Male,Female or Male,0,14,True,female or male
5,I1,row-0568-s007,qualification,"Analytical, trustworthy and very responsible",Analytical,0,10,True,analytical
6,I2,row-0317-s009,qualification,Good interpersonal skills,interpersonal,5,18,True,interpersonal
7,P,row-0397-s005,qualification,"Proficient in MS apps particularly MS Excel (pivot table, vlookup etc.) and data visualization tools",MS,14,16,True,ms
8,P,row-0029-s009,qualification,Send a direct message to Ms. Mary Claire Limpiada (Owner) for the schedule of interview.,Ms,25,27,True,ms
9,U,row-0012-s002,body_other,Male,Male,0,4,True,male


In [5]:
tax = taxonomy.set_index("code")
lines = [
    "# Codebook v0.3 (pilot) - gender expressions in PESO-linked job advertisements",
    "",
    "Status: proposal for review. Labels in the data are rule-derived candidates, not human judgments. "
    "The **expression code** records what the wording is; **final bias judgment** is a separate field left `pending`.",
    "",
]
for code_, d in CB.items():
    t = tax.loc[code_]
    lines += [f"## {code_} - {t['category']}", "", "| Element | Rule |", "|---|---|",
              f"| Definition | {t['definition']} |", f"| Apply when | {d['apply']} |", f"| Do not apply when | {d['not_apply']} |"]
    e = ex[ex["code"] == code_]
    if e.empty:
        lines.append("| Corpus example | none found in the 1,500 ads (see justification) |")
    for r in e.itertuples():
        ctx = {"job_title": "Job title", "qualification": "Qualification", "body_other": "Other body text"}[r.component]
        lines += [f"| Corpus example | {r.unit_id}: {ctx} -> \"{r.unit_text}\" |", f"| Span | `{r.span_in_text_norm}` (characters {r.span_start}-{r.span_end} of the normalized unit) |"]
        if r.why:
            lines.append(f"| Why | {r.why} |")
    lines += [f"| Primary classification | {t['primary_classification']} |", f"| Bias judgment prior | {t['bias_judgment_prior']} |", f"| Justification | {d['justification']} |", ""]
codebook_md = "\n".join(lines)
(OUT / "codebook_v0.3.md").write_text(codebook_md, encoding="utf-8")
display(Markdown(codebook_md))

# Codebook v0.3 (pilot) - gender expressions in PESO-linked job advertisements

Status: proposal for review. Labels in the data are rule-derived candidates, not human judgments. The **expression code** records what the wording is; **final bias judgment** is a separate field left `pending`.

## E1 - Gender restriction or preference

| Element | Rule |
|---|---|
| Definition | States that a gender is required or preferred. |
| Apply when | The span states a gender as an applicant requirement or preference ('Female', 'Male only', 'preferably female', 'for male'), and it functions as a condition on who may apply. |
| Do not apply when | The gender mention is a listing of both genders without a preference (G1). The gender word belongs to a job title (E2) or a pronoun (E3). |
| Corpus example | row-0038-s002: Qualification -> "Female" |
| Span | `female` (characters 0-6 of the normalized unit) |
| Why | Cashier; the standalone designation is presented as an applicant qualification |
| Primary classification | explicit |
| Bias judgment prior | Explicit |
| Justification | A standalone gender designation under qualifications is an eligibility condition. |

## E2 - Gender-marked occupational wording

| Element | Rule |
|---|---|
| Definition | Gender-specific occupational term, such as 'saleslady' or 'foreman'. |
| Apply when | A job title or role noun that itself marks gender ('saleslady', 'foreman', 'watchman', 'salesman'). |
| Do not apply when | The noun merely ends in -man/-men but is not a person term ('human', 'manager'). The word is a gender restriction on applicants (E1). |
| Corpus example | row-0176-s001: Job title -> "Saleslady" |
| Span | `saleslady` (characters 0-9 of the normalized unit) |
| Why | job title marks the role as female |
| Corpus example | row-0131-s001: Job title -> "Foreman" |
| Span | `foreman` (characters 0-7 of the normalized unit) |
| Why | job title with a male-marked noun |
| Primary classification | explicit |
| Bias judgment prior | Context-dependent |
| Justification | The gender is carried by the occupational term, not by a stated requirement; whether it is biased depends on how the ad uses it, so the judgment prior is context-dependent. |

## E3 - Generic gendered pronoun

| Element | Rule |
|---|---|
| Definition | Uses 'he' or 'she' as a generic reference to the position holder. |
| Apply when | 'he', 'she', 'his' or 'her' refers generically to the position holder, not to a named or specific person. |
| Do not apply when | The pronoun refers to a specific named person. Both pronouns are given together as a neutral pair (decision pending, see section 6). |
| Corpus example | row-0716-s006: Qualification -> "Submit Reports Daily & Weekly for sites under he’s area of responsibility" |
| Span | `he` (characters 46-48 of the normalized unit) |
| Why | 'he's area of responsibility' refers to the post holder |
| Primary classification | explicit |
| Bias judgment prior | Context-dependent |
| Justification | A gendered pronoun used for any holder of the role implies a default gender; judgment depends on context. |

## G1 - Non-restrictive gender listing

| Element | Rule |
|---|---|
| Definition | Mentions gender categories without a stated preference, e.g. 'Female/Male'. |
| Apply when | The text names both gender categories with no preference expressed ('Female/Male', 'Male or Female', 'he/she' as a listing). |
| Do not apply when | Only one gender is named (E1). A statement of inclusive eligibility without naming gender categories (INC). |
| Corpus example | row-0637-s003: Qualification -> "Female or Male" |
| Span | `female or male` (characters 0-14 of the normalized unit) |
| Why | both categories listed, no preference stated |
| Primary classification | none |
| Bias judgment prior | Not automatically biased |
| Justification | A listing does not state that either gender is preferred, so it is not claimed to equal 'Female applicants only'; it is not automatically biased. |

## I1 - Masculine-coded expression

| Element | Rule |
|---|---|
| Definition | Agentic or masculine-associated wording supported by a defined lexicon or contextual evidence. |
| Apply when | A word from the defined masculine-coded (agentic) lexicon is present. The code records the wording only; it does not say the ad is biased. |
| Do not apply when | The word is an ordinary role noun or topic ('analyst', 'analysis') - consider P. The word is only part of a technical term. |
| Corpus example | row-0568-s007: Qualification -> "Analytical, trustworthy and very responsible" |
| Span | `analytical` (characters 0-10 of the normalized unit) |
| Why | lexicon word in a qualification line |
| Primary classification | implicit |
| Bias judgment prior | Assess context |
| Justification | Lexicon match only; the judgment needs the full ad and is left to review. |

## I2 - Feminine-coded expression

| Element | Rule |
|---|---|
| Definition | Communal or feminine-associated wording supported by a defined lexicon or contextual evidence. |
| Apply when | A word from the defined feminine-coded (communal) lexicon is present. The code records the wording only. |
| Do not apply when | The word is an ordinary role noun ('support' in 'IT support') - consider P. |
| Corpus example | row-0317-s009: Qualification -> "Good interpersonal skills" |
| Span | `interpersonal` (characters 5-18 of the normalized unit) |
| Why | lexicon word in a qualification line |
| Primary classification | implicit |
| Bias judgment prior | Assess context |
| Justification | Lexicon match only; the judgment needs the full ad and is left to review. |

## INC - Inclusive expression

| Element | Rule |
|---|---|
| Definition | Explicitly expresses inclusive applicant eligibility. |
| Apply when | The ad explicitly states that applicants of any gender are eligible (for example 'regardless of gender', 'open to all genders'). |
| Do not apply when | Gender categories are listed as in 'Female/Male' (G1). General welcoming wording that does not mention gender ('open to all college graduates'). |
| Corpus example | none found in the 1,500 ads (see justification) |
| Primary classification | none |
| Bias judgment prior | Not biased on its own |
| Justification | In the v0.2 notebooks this code covered 'Female/Male' listings; those are now G1. No corpus unit with explicit all-gender eligibility wording was found by the probe in section 4, so no example is given. |

## P - Polysemantic / contextual near miss

| Element | Rule |
|---|---|
| Definition | Potential lexical match that is non-biased in context. |
| Apply when | A word that matches the gender lexicon but is not about gender in this context ('MS Office', the honorific 'Ms.' for a contact person). |
| Do not apply when | The word refers to the applicant's gender or marks the role by gender. |
| Corpus example | row-0397-s005: Qualification -> "Proficient in MS apps particularly MS Excel (pivot table, vlookup etc.) and data visualization tools" |
| Span | `ms` (characters 14-16 of the normalized unit) |
| Why | 'MS' here is Microsoft |
| Corpus example | row-0029-s009: Qualification -> "Send a direct message to Ms. Mary Claire Limpiada (Owner) for the schedule of interview." |
| Span | `ms` (characters 25-27 of the normalized unit) |
| Why | honorific for a contact person |
| Primary classification | none |
| Bias judgment prior | Not biased |
| Justification | Lexical match with no gender meaning in context. |

## U - Unclear expression

| Element | Rule |
|---|---|
| Definition | Cannot be classified reliably with available context. |
| Apply when | A gender word is present but the rules or the available unit text cannot place it (for example a gender word in a unit with no clear function, or a possible technical term). |
| Do not apply when | The span fits another code. Use U rather than guessing. |
| Corpus example | row-0012-s002: Other body text -> "Male" |
| Span | `male` (characters 0-4 of the normalized unit) |
| Why | a whole-unit gender word in a block without a qualification header; probably E1, not decided |
| Corpus example | row-0692-s004: Qualification -> "In depth knowledge of the four network types (LAN, WAN, MAN, and GAN)" |
| Span | `man` (characters 56-59 of the normalized unit) |
| Why | 'MAN' as a network type; probably P, not decided |
| Primary classification | unclear |
| Bias judgment prior | Requires review |
| Justification | Kept for review; the judgment prior is 'requires review'. |


## 3. Crosswalk: v0.2 candidates to v0.3 expression codes

Mapping is deterministic and does not look at the ad. It changes the **code names and the judgment fields**, not which spans were flagged.

| v0.2 candidate | v0.3 code | Note |
|---|---|---|
| E1 | E1 | unchanged |
| E2 | E2 | unchanged; judgment prior is now context-dependent |
| E4_or_E5 | E3 | generic pronoun; v0.2 had not chosen between 'generic he' and 'generic she' |
| INC (listing) | **G1** | the 'Female/Male' listings; v0.2 called these 'Inclusive' |
| I1, I2 | I1, I2 | unchanged; judgment prior 'assess context' |
| P | P | unchanged |
| needs_review | U | with a reason code |
| none (rejected by rule) | dropped | not a span |

New **INC** has no counterpart in v0.2 and no corpus instance, so it receives no candidates from this mapping.

In [6]:
MAP = {"E1": "E1", "E2": "E2", "E4_or_E5": "E3", "INC": "G1", "I1": "I1", "I2": "I2", "P": "P", "needs_review": "U", "none": None}
cw = spans.copy()
cw["expression_code"] = cw["v02_candidate"].map(MAP)
cw = cw[cw["expression_code"].notna()].copy()
cw["expression_category"] = cw["expression_code"].map(tax["category"])
cw["primary_classification"] = cw["expression_code"].map(tax["primary_classification"])
cw["bias_judgment_prior"] = cw["expression_code"].map(tax["bias_judgment_prior"])
cw["final_bias_judgment"] = "pending"

def reason(r):
    if r["expression_code"] != "U":
        return ""
    whole = str(r["text_masked"]).strip().lower() == str(r["span_text"]).strip().lower()
    if whole:
        return "gender word is the whole unit; unit is in " + r["component"] + ", not a qualification line"
    return "gender word inside a longer unit; rules could not place it"
cw["u_reason"] = cw.apply(reason, axis=1)
cols = ["unit_id", "record_id", "component", "text_masked", "span_text", "span_start", "span_end", "span_offset_valid", "raw_start", "raw_end",
        "v02_candidate", "expression_code", "expression_category", "primary_classification", "bias_judgment_prior", "final_bias_judgment", "review_status", "review_flag", "u_reason", "notes"]
cw[cols].to_csv(OUT / "crosswalk_v02_to_v03.csv", index=False, encoding="utf-8-sig")

summ = (cw.groupby(["v02_candidate", "expression_code", "primary_classification", "bias_judgment_prior"])
          .agg(span_rows=("unit_id", "size"), units=("unit_id", "nunique"), ads=("record_id", "nunique")).reset_index())
summ.to_csv(OUT / "crosswalk_summary.csv", index=False, encoding="utf-8-sig")
display(summ)
print("v0.2 rows:", len(spans), "-> v0.3 rows:", len(cw), "(dropped as rejected by rule:", int((spans["v02_candidate"] == "none").sum()), ")")
display(cw.loc[cw["expression_code"] == "U", ["unit_id", "component", "span_text", "text_masked", "u_reason"]].reset_index(drop=True))

,v02_candidate,expression_code,primary_classification,bias_judgment_prior,span_rows,units,ads
0,E1,E1,explicit,Explicit,317,312,311
1,E2,E2,explicit,Context-dependent,40,40,32
2,E4_or_E5,E3,explicit,Context-dependent,5,3,2
3,I1,I1,implicit,Assess context,350,324,240
4,I2,I2,implicit,Assess context,379,324,275
5,INC,G1,none,Not automatically biased,99,99,98
6,P,P,none,Not biased,11,11,11
7,needs_review,U,unclear,Requires review,16,16,16


v0.2 rows: 1218 -> v0.3 rows: 1217 (dropped as rejected by rule: 1 )


,unit_id,component,span_text,text_masked,u_reason
0,row-0012-s002,body_other,Male,Male,"gender word is the whole unit; unit is in body_other, not a qualification line"
1,row-0013-s002,body_other,Male,Male,"gender word is the whole unit; unit is in body_other, not a qualification line"
2,row-0014-s002,body_other,Female,Female,"gender word is the whole unit; unit is in body_other, not a qualification line"
3,row-0015-s002,body_other,Female,Female,"gender word is the whole unit; unit is in body_other, not a qualification line"
4,row-0016-s002,body_other,Male,Male,"gender word is the whole unit; unit is in body_other, not a qualification line"
5,row-0018-s002,body_other,Female,Female,"gender word is the whole unit; unit is in body_other, not a qualification line"
6,row-0056-s003,body_other,Male,Male,"gender word is the whole unit; unit is in body_other, not a qualification line"
7,row-0328-s006,qualification,male,"18-25 years old, male at least 5'5"" can work under pressure",gender word inside a longer unit; rules could not place it
8,row-0651-s003,qualification,female,"With pleasing personality, female and single",gender word inside a longer unit; rules could not place it
9,row-0652-s003,qualification,female,"With pleasing personality, female and single",gender word inside a longer unit; rules could not place it


## 4. Probe: wording the lexicon does not cover

The v0.2 lexicon is a short list, so the rules cannot be assumed to be complete. This probe scans every unit for gender words and reports any that no span covers (whole-word match). It also lists units that name both 'male' and 'female' yet carry only E1 spans, which are possible G1 listings that the listing wording missed. It **does not change labels**; it lists what a reviewer would meet.

In [7]:
GW = re.compile(r"\b(?:male|female|males|females|men|women|man|woman|boy|girl|lady|ladies|gentleman|he|she|his|her|him)\b", re.I)
INCLUSIVE = re.compile(r"regardless of (?:gender|sex)|any gender|all genders|open to (?:both|all genders)|both sexes|gender[- ]neutral|equal opportunity", re.I)
cov = spans.groupby("unit_id")["span_text"].apply(lambda s: set(re.findall(r"\w+", " ".join(s.astype(str).str.lower()))))
rows = []
for u in units.itertuples():
    t = str(u.text_masked)
    covered = cov.get(u.unit_id, set())
    for m in GW.finditer(t):
        if m.group(0).lower() not in covered:
            rows.append({"unit_id": u.unit_id, "component": u.component, "term": m.group(0), "text_masked": t, "unit_has_other_span": u.unit_id in cov.index})
probe = pd.DataFrame(rows, columns=["unit_id", "component", "term", "text_masked", "unit_has_other_span"])
probe["term_lc"] = probe["term"].str.lower()
probe.drop(columns="term_lc").to_csv(OUT / "lexicon_gap_probe.csv", index=False, encoding="utf-8-sig")
print(f"{len(probe)} uncovered gender-word hits in {probe['unit_id'].nunique()} units")
display(probe["term_lc"].value_counts().rename_axis("term").to_frame("hits").head(15))
both = probe[probe["text_masked"].str.contains(r"\bmale\b", case=False) & probe["text_masked"].str.contains(r"\bfemale\b", case=False)]
print("Uncovered units naming both 'male' and 'female' (possible G1):")
display(both[["unit_id", "text_masked"]].drop_duplicates())
inc_hits = units[units["text_masked"].str.contains(INCLUSIVE)]
print(f"Units matching an explicit all-gender eligibility phrase (possible INC): {len(inc_hits)}")
display(inc_hits[["unit_id", "text_masked"]])

e1_only = cw[cw["expression_code"] == "E1"].groupby("unit_id")["text_masked"].first()
both_e1 = e1_only[e1_only.str.contains(r"\bmale\b", case=False) & e1_only.str.contains(r"\bfemale\b", case=False)]
glisting = cw[cw["expression_code"] == "G1"]["unit_id"]
both_e1 = both_e1[~both_e1.index.isin(glisting)]
print(f"Units naming both 'male' and 'female' whose spans are E1, not G1 (possible mislabelled listings): {len(both_e1)}")
display(both_e1.rename("text_masked").reset_index())
both_e1.rename("text_masked").reset_index().to_csv(OUT / "possible_g1_labelled_e1.csv", index=False, encoding="utf-8-sig")
display(probe[["unit_id", "term", "text_masked"]].drop_duplicates("text_masked").head(25).reset_index(drop=True))

0 uncovered gender-word hits in 0 units


,hits
term,


Uncovered units naming both 'male' and 'female' (possible G1):


,unit_id,text_masked


Units matching an explicit all-gender eligibility phrase (possible INC): 0


,unit_id,text_masked
unit_id,,


Units naming both 'male' and 'female' whose spans are E1, not G1 (possible mislabelled listings): 5


,unit_id,text_masked
0,row-0109-s011,Must be atleast one meter and sixty-two centimeters (1.62 cm) in height for Male and one meter and fifty-s...
1,row-0520-s002,Male and Female
2,row-0542-s003,"At least 5'4"" height for Male and 5'0"" for Female"
3,row-0707-s002,Female and Male
4,row-1365-s002,"Male, Female, LGBT"


,unit_id,term,text_masked


## 5. Validation

In [8]:
checks = []
def rec(name, ok, detail=""):
    checks.append({"check": name, "passed": bool(ok), "detail": detail})

rec("every v0.2 candidate code has a v0.3 mapping", set(spans["v02_candidate"]) <= set(MAP), str(sorted(set(spans["v02_candidate"]))))
rec("no span rows lost except rule-rejected ones", len(cw) == len(spans) - int((spans["v02_candidate"] == "none").sum()))
rec("every code has the six codebook elements", all(all(d.get(k) for k in ("apply", "not_apply", "justification")) and c in tax.index for c, d in CB.items()) and set(CB) == set(tax.index))
rec("every code except INC has at least one corpus example", all(len(ex[ex["code"] == c]) >= 1 for c in tax.index if c != "INC"), "INC: none found")
rec("every example is a v0.2 candidate produced by the rules", bool(ex["found_as_v02_candidate"].all()))
rec("every example span reproduces its text in text_norm", all(str(a).lower() == str(b).lower() for a, b in zip(ex["span"], ex["span_in_text_norm"])))
rec("every code has a bias_judgment_prior and primary classification", tax[["bias_judgment_prior", "primary_classification"]].notna().all().all())
rec("final_bias_judgment is pending for every span", (cw["final_bias_judgment"] == "pending").all())
rec("expression code and judgment are separate columns", {"expression_code", "bias_judgment_prior", "final_bias_judgment"} <= set(cw.columns))
rec("G1 count equals the v0.2 INC count", int((cw["expression_code"] == "G1").sum()) == int((spans["v02_candidate"] == "INC").sum()))
rec("no span is assigned the new INC code", int((cw["expression_code"] == "INC").sum()) == 0)
rec("the Cashier example (row-0038) is the Female span in the Cashier ad", units.loc["row-0038-s001", "text_masked"] == "Cashier" and units.loc["row-0038-s002", "text_masked"] == "Female")
validation = pd.DataFrame(checks)
display(validation)
validation.to_csv(OUT / "validation_checks.csv", index=False)
assert validation["passed"].all(), validation[~validation["passed"]]

,check,passed,detail
0,every v0.2 candidate code has a v0.3 mapping,True,"['E1', 'E2', 'E4_or_E5', 'I1', 'I2', 'INC', 'P', 'needs_review', 'none']"
1,no span rows lost except rule-rejected ones,True,
2,every code has the six codebook elements,True,
3,every code except INC has at least one corpus example,True,INC: none found
4,every example is a v0.2 candidate produced by the rules,True,
5,every example span reproduces its text in text_norm,True,
6,every code has a bias_judgment_prior and primary classification,True,
7,final_bias_judgment is pending for every span,True,
8,expression code and judgment are separate columns,True,
9,G1 count equals the v0.2 INC count,True,


## 6. Decisions needed before this becomes the working taxonomy

These are listed, not decided, because each changes labels or counts.

1. **Replace v0.2 with v0.3?** If yes, `annotation-guidelines-gender-v0.2.md` is superseded by the codebook, and Steps 5, 5a and 6 would be re-run with the v0.3 codes. Until then they keep v0.2 codes.
2. **E2 judgment.** v0.2 treated gendered titles as explicit bias (decision D2). The pilot taxonomy gives E2 a context-dependent prior. The crosswalk keeps `primary_classification = explicit` for the expression type and moves the stance into the prior; confirm this reading.
3. **'his/her' and 'he/she' pairs.** The two 'he/she' units were in the v0.2 listing group (now G1), while the 'his/her' pair in row-0109-s004 was a pronoun candidate (now E3). Both are neutral pairs; one rule for both is needed.
4. **The 16 unclear spans.** Most are a whole-unit 'Male' or 'Female' in `body_other` (ads without a qualification header), 'female and single', and 'male at least 5'5"', which read as E1 by the codebook; two are 'MAN' as a network type (P). They stay U until a reviewer or an explicit extension of the E1 rule ('whole-unit gender word in any component') is approved.
5. **INC has no instance.** It stays in the taxonomy as defined, with zero candidates.
6. **Listing wording missed by the rules.** Section 4 lists units naming both 'male' and 'female' that carry E1 spans only (for example 'Male and Female'); they would be G1 under the codebook. Extending the listing wording would change counts and needs approval.

## 7. Test for overlapping classes (before freezing the taxonomy)

**Question.** Can two codes describe the same text? And does every code have an operational definition, positive and negative conditions, and a defensible route to one of the primary classifications `explicit`, `implicit`, `none` or `unclear`?

**Data.** Only the preprocessed units (`units.csv`) and the Step 5 spans carried into the crosswalk above. Nothing is simulated or typed in as an example; every row shown comes from the corpus.

**Method**
1. **Roles and routes.** Each code is given a role: *primary* (a lexical category that can carry bias: E1, E2, E3, I1, I2), *auxiliary* (G1, INC, P: they describe wording that is not treated as a bias category) or *review* (U). Each code routes to one primary classification.
2. **Test 1 - character overlap.** Do two spans with different codes cover the same characters of one unit?
3. **Test 2 - one wording, several codes.** Does the same wording receive more than one code across the corpus?
4. **Test 3 - operational conditions.** The positive and negative conditions of the codebook are written as simple tests on the span, its unit and its component. Each span is run through every code's conditions. A span is *single* if exactly one code's conditions hold, *overlap* if two or more hold, and *no fit* if none holds. The result is compared with the code the Step 5 rules assigned. The conditions are a first operationalisation of the codebook text, not validated rules.
5. **Test 4 - auxiliary labels.** Do G1 and P compete with the primary codes inside the same unit or advertisement?
6. **Test 5 - separation of expression and judgment** on gender-marked titles, using the corpus titles that carry the gendered noun.
7. **Test 6 - completion criterion** per code.

Nothing here changes the labels in Steps 5, 5a or 6.

In [9]:
ROLE = {"E1": "primary", "E2": "primary", "E3": "primary", "I1": "primary", "I2": "primary", "G1": "auxiliary", "INC": "auxiliary", "P": "auxiliary", "U": "review"}
ROUTES = {"explicit", "implicit", "none", "unclear"}
route_table = taxonomy[["code", "category", "primary_classification", "bias_judgment_prior"]].copy()
route_table.insert(1, "role", route_table["code"].map(ROLE))
route_table = route_table.rename(columns={"primary_classification": "route"})
display(route_table)

by_route = cw.groupby("primary_classification").agg(span_rows=("unit_id", "size"), units=("unit_id", "nunique")).reset_index()
display(by_route)

,code,role,category,route,bias_judgment_prior
0,E1,primary,Gender restriction or preference,explicit,Explicit
1,E2,primary,Gender-marked occupational wording,explicit,Context-dependent
2,E3,primary,Generic gendered pronoun,explicit,Context-dependent
3,G1,auxiliary,Non-restrictive gender listing,none,Not automatically biased
4,I1,primary,Masculine-coded expression,implicit,Assess context
5,I2,primary,Feminine-coded expression,implicit,Assess context
6,INC,auxiliary,Inclusive expression,none,Not biased on its own
7,P,auxiliary,Polysemantic / contextual near miss,none,Not biased
8,U,review,Unclear expression,unclear,Requires review


,primary_classification,span_rows,units
0,explicit,362,355
1,implicit,729,595
2,none,110,110
3,unclear,16,16


### Test 1 - do spans with different codes cover the same characters?

In [10]:
valid = cw[cw["span_offset_valid"] == True].dropna(subset=["span_start", "span_end"])
ov = []
for uid, g in valid.groupby("unit_id"):
    g = g.sort_values(["span_start", "span_end"]).reset_index(drop=True)
    for i in range(len(g)):
        for j in range(i + 1, len(g)):
            if g.loc[j, "span_start"] < g.loc[i, "span_end"] and g.loc[i, "expression_code"] != g.loc[j, "expression_code"]:
                ov.append({"unit_id": uid, "span_a": g.loc[i, "span_text"], "code_a": g.loc[i, "expression_code"], "span_b": g.loc[j, "span_text"], "code_b": g.loc[j, "expression_code"]})
char_overlap = pd.DataFrame(ov, columns=["unit_id", "span_a", "code_a", "span_b", "code_b"])
print(f"{len(valid):,} spans with valid offsets; spans with different codes that share characters: {len(char_overlap)}")
print("Units holding more than one code:", int((cw.groupby("unit_id")["expression_code"].nunique() > 1).sum()))
display(cw.groupby("unit_id")["expression_code"].apply(lambda s: "+".join(sorted(set(s)))).loc[lambda s: s.str.contains(r"\+")].value_counts().rename_axis("codes in one unit").to_frame("units"))

1,216 spans with valid offsets; spans with different codes that share characters: 0
Units holding more than one code: 56


,units
codes in one unit,
I1+I2,53
E3+I1,1
E3+I2,1
I1+P,1


### Test 2 - does one wording receive more than one code?

In [11]:
w = cw.assign(wording=cw["span_text"].str.lower()).groupby("wording")["expression_code"].agg(lambda s: sorted(set(s)))
multi = w[w.map(len) > 1]
mrows = []
for wd, codes in multi.items():
    sub = cw[cw["span_text"].str.lower() == wd]
    mrows.append({"wording": wd, "codes": "+".join(codes), **{c: int((sub["expression_code"] == c).sum()) for c in sorted(cw["expression_code"].unique())}})
multi_wording = pd.DataFrame(mrows)
print(f"{len(w)} distinct wordings; {len(multi_wording)} receive more than one code")
display(multi_wording.loc[:, (multi_wording != 0).any(axis=0)] if len(multi_wording) else multi_wording)

107 distinct wordings; 3 receive more than one code


,wording,codes,E1,E2,U
0,female,E1+U,100,0,8
1,male,E1+U,192,0,6
2,man,E2+U,0,2,2


### Test 3 - operational conditions per code, applied to every span

| Code | Positive condition (all must hold) | Negative condition (any one removes it) |
|---|---|---|
| E1 | span contains a person-gender word; the unit is not a job title, or the span carries a requirement cue ("preferably", "only", "for", "applicant") | the unit lists both genders (G1); the gender word is attached to a requirement for each gender ("for Male ... for Female", "(Female)" after a measure); the word is the acronym "MAN" |
| E2 | the span is an occupational noun ending in -man, -men, -woman, -lady, -girl or -boy (in any component), or a bare person-gender word inside a job title | a requirement cue is present (then E1); the word is a known non-gender form (e.g. "human"); the word is the acronym "MAN" |
| E3 | span is he, she, his, her or him | the pronoun is part of a pair ("his/her", "he/she") |
| G1 | the unit names two genders joined by and, or, "/", "," or "&" (he/she and his/her pairs included) | the pair sits inside a requirement for each gender |
| I1, I2 | the span is a word from the masculine-coded / feminine-coded lexicon already in use | none defined in v0.3 |
| P | the span is "MS" / "Ms" (honorific or Microsoft), or "MAN" inside a list of capitalised acronyms | none defined |
| P (role noun) | the I1/I2 word is also an ordinary role noun: analyst, analysis, support, leader | none |
| INC | the unit states that applicants of any gender are eligible | the unit only lists gender categories (G1) |
| U | no other code's conditions hold | - |

The P role-noun row is shown separately because it is a *candidate* overlap with I1/I2: the codebook says to "consider P" but gives no rule that decides.

In [12]:
GN = r"(?:male|female|males|females|men|women|man|woman|boy|girl|lady|ladies|gentleman|lalaki|babae)"
GN_RE = re.compile(rf"\b{GN}\b", re.I)
PR_RE = re.compile(r"^(?:he|she|his|her|him)$", re.I)
PR_FULL = re.compile(r"^(?:he|she|his|her|him)(?:\s*[/,&]\s*(?:he|she|his|her|him))*$", re.I)
CUE = re.compile(r"\b(?:preferably|only|applicants?|for)\b", re.I)
PAIR = re.compile(r"\b(?:male|female|men|women|he|she|his|her)\b\s*(?:/|,|&|\band\b|\bor\b)\s*(?:male|female|men|women|he|she|his|her|lgbt\w*)\b", re.I)
COND = re.compile(r"\bfor\s+(?:male|female|men|women)\b.*\bfor\s+(?:male|female|men|women)\b|\((?:male|female)\)", re.I)
INCL = re.compile(r"regardless of (?:gender|sex)|any gender|all genders|both sexes|gender[- ]neutral", re.I)
SUFFIX = re.compile(r"(?:man|men|woman|lady|girl|boy)$", re.I)
NOT_GENDER = {"human", "german", "roman", "norman"}
ROLE_NOUNS = {"analyst", "analysis", "support", "leader"}
I1_SET = set(cw.loc[cw["expression_code"] == "I1", "span_text"].str.lower())
I2_SET = set(cw.loc[cw["expression_code"] == "I2", "span_text"].str.lower())
ACRO = re.compile(r"\b[A-Z]{3}\b")


def positives(text, unit, comp):
    low = str(text).strip().lower()
    is_gn = bool(GN_RE.search(str(text)))
    is_pr = bool(PR_FULL.match(low))
    single_pr = bool(PR_RE.match(low))
    pair = bool(PAIR.search(str(unit)))
    cond = bool(COND.search(str(unit)))
    cue = bool(CUE.search(low))
    is_title = comp == "job_title"
    acro_man = text == "MAN" and len(ACRO.findall(str(unit))) >= 2
    bare = bool(re.fullmatch(GN, low))
    out = set()
    if is_gn and not pair and not cond and not acro_man and (not is_title or cue):
        out.add("E1")
    if not cue and not acro_man and low not in NOT_GENDER and text != "MAN" and ((is_title and bare) or (SUFFIX.search(low) and not bare)):
        out.add("E2")
    if single_pr and not pair:
        out.add("E3")
    if (is_gn or is_pr) and pair and not cond:
        out.add("G1")
    if low in I1_SET:
        out.add("I1")
    if low in I2_SET:
        out.add("I2")
    if low == "ms" or acro_man:
        out.add("P")
    if low in ROLE_NOUNS:
        out.add("P")
    if INCL.search(str(unit)):
        out.add("INC")
    return out


cw["predicted_codes"] = [positives(t, u, c) for t, u, c in zip(cw["span_text"], cw["text_masked"], cw["component"])]
cw["n_pred"] = cw["predicted_codes"].map(len)
cw["pred_status"] = cw["n_pred"].map({0: "no fit", 1: "single"}).fillna("overlap")
cw["assigned_in_pred"] = [a in p for a, p in zip(cw["expression_code"], cw["predicted_codes"])]
cw["pred_str"] = cw["predicted_codes"].map(lambda s: "+".join(sorted(s)))
status = pd.crosstab(cw["expression_code"], cw["pred_status"], margins=True)
agree = cw.groupby("expression_code")["assigned_in_pred"].agg(spans="size", assigned_code_satisfies_its_conditions="sum").reset_index()
agree["share"] = (agree["assigned_code_satisfies_its_conditions"] / agree["spans"]).map("{:.1%}".format)
print("Status of each span under the operational conditions (rows: code assigned by the Step 5 rules):")
display(status)
print("Does the assigned code satisfy its own positive and negative conditions?")
display(agree)

Status of each span under the operational conditions (rows: code assigned by the Step 5 rules):


pred_status,no fit,overlap,single,All
expression_code,,,,
E1,4,0,313,317
E2,1,0,39,40
E3,0,0,5,5
G1,0,0,99,99
I1,0,66,284,350
I2,0,43,336,379
P,0,0,11,11
U,1,0,15,16
All,6,109,1102,1217


Does the assigned code satisfy its own positive and negative conditions?


,expression_code,spans,assigned_code_satisfies_its_conditions,share
0,E1,317,307,96.8%
1,E2,40,39,97.5%
2,E3,5,1,20.0%
3,G1,99,99,100.0%
4,I1,350,350,100.0%
5,I2,379,379,100.0%
6,P,11,11,100.0%
7,U,16,0,0.0%


In [13]:
print("Overlapping spans (two or more codes' conditions hold):")
ovl = (cw[cw["pred_status"] == "overlap"].groupby(["expression_code", "pred_str"])
         .agg(span_rows=("unit_id", "size"), wordings=("span_text", lambda s: ", ".join(sorted(set(s.str.lower()))[:6]))).reset_index())
display(ovl)
print("Spans where the conditions point to a different code than the one assigned (single prediction):")
dis = (cw[(cw["pred_status"] == "single") & (~cw["assigned_in_pred"])]
         .groupby(["expression_code", "pred_str"]).agg(span_rows=("unit_id", "size"), units=("unit_id", "nunique"),
                                                     example=("text_masked", lambda s: s.iloc[0][:80])).reset_index())
display(dis)
print("Spans no code's conditions fit:")
nofit = cw[cw["pred_status"] == "no fit"][["unit_id", "expression_code", "component", "span_text", "text_masked"]]
display(nofit.drop_duplicates(["span_text", "text_masked"]).reset_index(drop=True))

Overlapping spans (two or more codes' conditions hold):


,expression_code,pred_str,span_rows,wordings
0,I1,E2+I1,3,leadman
1,I1,I1+P,63,"analysis, analyst, leader"
2,I2,I2+P,43,support


Spans where the conditions point to a different code than the one assigned (single prediction):


,expression_code,pred_str,span_rows,units,example
0,E1,G1,6,3,Male and Female
1,E3,G1,4,2,"Must have passed the neuro-psychiatric evaluation, medical examination and drug"
2,U,E1,13,13,Male
3,U,P,2,2,"In depth knowledge of the four network types (LAN, WAN, MAN, and GAN)"


Spans no code's conditions fit:


,unit_id,expression_code,component,span_text,text_masked
0,row-0109-s011,E1,qualification,for Male,Must be atleast one meter and sixty-two centimeters (1.62 cm) in height for Male and one meter and fifty-s...
1,row-0109-s011,E1,qualification,for Female,Must be atleast one meter and sixty-two centimeters (1.62 cm) in height for Male and one meter and fifty-s...
2,row-0542-s003,E1,qualification,for Male,"At least 5'4"" height for Male and 5'0"" for Female"
3,row-0542-s003,E1,qualification,for Female,"At least 5'4"" height for Male and 5'0"" for Female"
4,row-0989-s002,U,qualification,Female,At least 5’4”(Female) in height
5,row-1441-s001,E2,job_title,MAN,TP MAN Production


### Test 3b - are there units where the E2 conditions hold but no span was assigned?

The E2 condition is applied to every job title in the corpus, not only to titles the Step 5 rules flagged. This checks whether the codebook's positive condition finds gender-marked titles that the pre-annotation lexicon did not.

In [14]:
titles = units[units["component"] == "job_title"].copy()
titles["e2_condition"] = [any("E2" in positives(tok, t, "job_title") for tok in re.findall(r"[A-Za-z]+", t)) for t in titles["text_masked"]]
flagged_titles = set(cw.loc[cw["expression_code"] == "E2", "unit_id"])
titles["has_e2_span"] = titles["unit_id"].isin(flagged_titles)
print(f"{len(titles):,} job titles; E2 conditions hold in {int(titles['e2_condition'].sum())}; of these {int((titles['e2_condition'] & titles['has_e2_span']).sum())} carry an E2 span and {int((titles['e2_condition'] & ~titles['has_e2_span']).sum())} carry none.")
missed_e2 = titles[titles["e2_condition"] & ~titles["has_e2_span"]]
nouns = missed_e2["text_masked"].str.extract(r"(\w*(?:man|men|woman|lady|girl|boy))\b", flags=re.I)[0].str.lower().value_counts().rename_axis("gender-marked noun in title").to_frame("titles")
display(nouns)
print("E2-flagged titles by noun:")
display(titles[titles["has_e2_span"]]["text_masked"].str.extract(r"(\w*(?:man|men|woman|lady|girl|boy))\b", flags=re.I)[0].str.lower().value_counts().rename_axis("noun").to_frame("titles"))
print("E2 spans outside job titles (component):")
display(cw[cw["expression_code"] == "E2"].groupby("component").size().rename("spans").to_frame())
missed_e2[["unit_id", "record_id", "text_masked"]].to_csv(OUT / "e2_condition_titles_without_span.csv", index=False, encoding="utf-8-sig")

1,497 job titles; E2 conditions hold in 51; of these 29 carry an E2 span and 22 carry none.


,titles
gender-marked noun in title,
warehouseman,5
leadman,3
tireman,3
steelman,3
lubeman,1
instrumentman,1
stockman,1
loadingmen,1
ductmen,1


E2-flagged titles by noun:


,titles
noun,
foreman,15
salesman,10
man,2
watchman,1
saleslady,1
boy,1


E2 spans outside job titles (component):


,spans
component,
job_title,30
qualification,10


### Test 4 - do auxiliary labels (G1, P) compete with primary codes?

In [15]:
prim = cw[cw["expression_code"].map(ROLE) == "primary"]
rows = []
for aux in ("G1", "P", "U"):
    a = cw[cw["expression_code"] == aux]
    rows.append({"auxiliary code": aux, "spans": len(a), "units": a["unit_id"].nunique(),
                 "units that also hold a primary-code span": len(set(a["unit_id"]) & set(prim["unit_id"])),
                 "ads": a["record_id"].nunique(), "ads that also hold a primary-code span": len(set(a["record_id"]) & set(prim["record_id"])),
                 "ads with this code and no primary-code span": len(set(a["record_id"]) - set(prim["record_id"]))})
aux_tab = pd.DataFrame(rows)
display(aux_tab)
rt = cw.groupby("record_id")["primary_classification"].agg(lambda s: set(s))
def ad_route(s):
    if "explicit" in s: return "explicit"
    if "implicit" in s: return "implicit"
    if "unclear" in s: return "unclear"
    return "none"
ad_routes = rt.map(ad_route).value_counts().rename_axis("ad route (highest of its span routes)").to_frame("ads")
display(ad_routes)

,auxiliary code,spans,units,units that also hold a primary-code span,ads,ads that also hold a primary-code span,ads with this code and no primary-code span
0,G1,99,99,0,98,27,71
1,P,11,11,1,11,10,1
2,U,16,16,0,16,7,9


,ads
ad route (highest of its span routes),
implicit,344
explicit,340
none,72
unclear,9


### Test 5 - expression code kept apart from the judgment: gender-marked titles

For a title such as "Saleslady" the codebook records one lexical code (E2) and leaves the bias judgment to a separate field. The test below shows, for each gender-marked title noun in the corpus, which codes' conditions hold; a second code (for example an eligibility restriction, E1) should not hold on the same text.

In [16]:
e2 = cw[cw["expression_code"] == "E2"].copy()
e2["noun"] = e2["span_text"].str.lower()
t5 = (e2.groupby(["noun", "component"]).agg(spans=("unit_id", "size"), example_unit=("unit_id", "first"), example_text=("text_masked", "first"),
                                          expression_code=("expression_code", "first"), primary_classification=("primary_classification", "first"),
                                          bias_judgment_prior=("bias_judgment_prior", "first"), final_bias_judgment=("final_bias_judgment", "first"),
                                          conditions_that_hold=("pred_str", lambda s: " | ".join(sorted(set(s))))).reset_index())
display(t5)
print("E2 spans on which E1 conditions also hold:", int(e2["predicted_codes"].map(lambda p: "E1" in p).sum()))
print("Corpus titles containing saleslady / salesgirl / salesboy / saleswoman:")
display(units[(units["component"] == "job_title") & units["text_masked"].str.contains(r"sales(?:lady|girl|boy|woman)", case=False)][["unit_id", "text_masked"]])

,noun,component,spans,example_unit,example_text,expression_code,primary_classification,bias_judgment_prior,final_bias_judgment,conditions_that_hold
0,boy,job_title,1,row-0461-s001,Farm Boy,E2,explicit,Context-dependent,pending,E2
1,foreman,job_title,15,row-0131-s001,Foreman,E2,explicit,Context-dependent,pending,E2
2,foreman,qualification,7,row-0138-s003,With at least three (3) years' work experience as Foreman,E2,explicit,Context-dependent,pending,E2
3,man,job_title,2,row-1384-s001,Steel man,E2,explicit,Context-dependent,pending,| E2
4,saleslady,job_title,1,row-0176-s001,Saleslady,E2,explicit,Context-dependent,pending,E2
5,salesman,job_title,10,row-0626-s001,Salesman,E2,explicit,Context-dependent,pending,E2
6,salesman,qualification,3,row-0948-s003,With at least 2 years experience as salesman,E2,explicit,Context-dependent,pending,E2
7,watchman,job_title,1,row-0083-s001,Watchman,E2,explicit,Context-dependent,pending,E2


E2 spans on which E1 conditions also hold: 0
Corpus titles containing saleslady / salesgirl / salesboy / saleswoman:


,unit_id,text_masked
unit_id,,
row-0176-s001,row-0176-s001,Saleslady


### Test 6 - completion criterion per code

Elements checked: operational definition; positive condition; negative condition; route to `explicit`/`implicit`/`none`/`unclear`; at least one corpus example. Test results: spans assigned, share whose assigned code satisfies its own conditions, spans in overlap, spans where the conditions point elsewhere. A code is **not** called ready on the elements alone; the "test finding" column states what the data showed.

In [17]:
rows = []
for code_ in taxonomy["code"]:
    d, sub = CB[code_], cw[cw["expression_code"] == code_]
    n = len(sub)
    ok = int(sub["assigned_in_pred"].sum()) if n else 0
    rows.append({
        "code": code_, "role": ROLE[code_], "route": tax.loc[code_, "primary_classification"],
        "definition": bool(tax.loc[code_, "definition"]), "positive_condition": bool(d["apply"]), "negative_condition": bool(d["not_apply"]),
        "route_valid": tax.loc[code_, "primary_classification"] in ROUTES,
        "corpus_examples": int((ex["code"] == code_).sum()),
        "spans_assigned": n, "assigned_satisfies_conditions": ok,
        "in_overlap": int((sub["pred_status"] == "overlap").sum()), "conditions_point_elsewhere": int(((sub["pred_status"] == "single") & ~sub["assigned_in_pred"]).sum()),
        "no_fit": int((sub["pred_status"] == "no fit").sum()),
    })
crit = pd.DataFrame(rows)
display(crit)

,code,role,route,definition,positive_condition,negative_condition,route_valid,corpus_examples,spans_assigned,assigned_satisfies_conditions,in_overlap,conditions_point_elsewhere,no_fit
0,E1,primary,explicit,True,True,True,True,1,317,307,0,6,4
1,E2,primary,explicit,True,True,True,True,2,40,39,0,0,1
2,E3,primary,explicit,True,True,True,True,1,5,1,0,4,0
3,G1,auxiliary,none,True,True,True,True,1,99,99,0,0,0
4,I1,primary,implicit,True,True,True,True,1,350,350,66,0,0
5,I2,primary,implicit,True,True,True,True,1,379,379,43,0,0
6,INC,auxiliary,none,True,True,True,True,0,0,0,0,0,0
7,P,auxiliary,none,True,True,True,True,2,11,11,0,0,0
8,U,review,unclear,True,True,True,True,2,16,0,0,15,1


## 7a. Validation of the overlap test

In [18]:
checks2 = []
def rec2(name, ok, detail=""):
    checks2.append({"check": name, "passed": bool(ok), "detail": detail})

rec2("every code has a role and a route in {explicit, implicit, none, unclear}", set(route_table["route"]) <= ROUTES and route_table["role"].notna().all(), str(sorted(set(route_table["route"]))))
rec2("every crosswalk span row was tested", len(cw) == int(status.loc["All", "All"]), f"{len(cw):,} spans")
rec2("span statuses sum to the span total", int(cw["pred_status"].value_counts().sum()) == len(cw))
rec2("all test inputs come from the preprocessed units", cw["unit_id"].isin(units.index).all() and titles["unit_id"].isin(units.index).all())
rec2("every wording in the multi-code table is a span text from the corpus", set(multi_wording["wording"]) <= set(cw["span_text"].str.lower()) if len(multi_wording) else True)
rec2("every example in the E2 table exists in the units", all(t5["example_unit"].isin(units.index)))
rec2("every code has definition, positive and negative condition", bool(crit[["definition", "positive_condition", "negative_condition"]].all().all()))
rec2("route counts sum to the span total", int(by_route["span_rows"].sum()) == len(cw))
rec2("final_bias_judgment still pending everywhere (no judgment made in this test)", (cw["final_bias_judgment"] == "pending").all())
val2 = pd.DataFrame(checks2)
display(val2)

route_table.to_csv(OUT / "overlap_test_roles_routes.csv", index=False)
cw[["unit_id", "record_id", "component", "span_text", "text_masked", "expression_code", "pred_str", "pred_status", "assigned_in_pred"]].to_csv(OUT / "overlap_test_span_results.csv", index=False, encoding="utf-8-sig")
status.to_csv(OUT / "overlap_test_status_by_code.csv")
ovl.to_csv(OUT / "overlap_test_overlaps.csv", index=False, encoding="utf-8-sig")
dis.to_csv(OUT / "overlap_test_assigned_vs_conditions.csv", index=False, encoding="utf-8-sig")
nofit.drop_duplicates(["span_text", "text_masked"]).to_csv(OUT / "overlap_test_no_fit.csv", index=False, encoding="utf-8-sig")
multi_wording.to_csv(OUT / "overlap_test_wording_multi_code.csv", index=False, encoding="utf-8-sig")
aux_tab.to_csv(OUT / "overlap_test_auxiliary.csv", index=False, encoding="utf-8-sig")
t5.to_csv(OUT / "overlap_test_e2_separation.csv", index=False, encoding="utf-8-sig")
crit.to_csv(OUT / "completion_criterion.csv", index=False)
val2.to_csv(OUT / "overlap_test_validation_checks.csv", index=False)
assert val2["passed"].all(), val2[~val2["passed"]]

,check,passed,detail
0,"every code has a role and a route in {explicit, implicit, none, unclear}",True,"['explicit', 'implicit', 'none', 'unclear']"
1,every crosswalk span row was tested,True,"1,217 spans"
2,span statuses sum to the span total,True,
3,all test inputs come from the preprocessed units,True,
4,every wording in the multi-code table is a span text from the corpus,True,
5,every example in the E2 table exists in the units,True,
6,"every code has definition, positive and negative condition",True,
7,route counts sum to the span total,True,
8,final_bias_judgment still pending everywhere (no judgment made in this test),True,
